# Figure 2 — causal-variant genetics across expression & splicing QTLs


This notebook reproduces **Figure 2** and its supplemental. The figure summarizes
properties of statistically fine-mapped causal variants (SuSiE, PIP > 0.9) across
molecular QTL assays in *All of Us*: their allele-frequency spectrum, ancestry
enrichment, molecular effect sizes, and their relationship to expression-outlier
and dosage-like phenotypes.

Figure 2 uses a **prep-then-panels** pattern: `01_prep_data.R` reads the raw
inputs once and derives every per-panel data frame into `Fig2/data/*.rds`; each
panel then reads its own prepared frame. (Those prepared `.rds` are also committed,
so the panels can run even without the raw inputs.)


## How to run

* **Kernel:** R (IRkernel). **Working directory:** the repository root (the folder that contains `Fig2/`).
* Run the cells top to bottom. Each panel caches its ggplot object to `Fig2/objects/`, and the assembly cell reads those back.
* Re-running a single panel cell rebuilds just that panel; re-run the assembly cell to recompose.


### Inputs

> **Controlled-access.** The raw inputs derive from *All of Us* controlled-access
> data and are not in this repository. The prep step reads them **by manifest key**
> (`Fig2/inputs.tsv`) from the staging root `FIG2_INPUT` (default the cwd; the files
> live under `input_data/`); the derived per-panel `Fig2/data/*.rds` are committed,
> so the panel cells below run from those even if the raw inputs are absent.


## Setup

`00_setup.R` is the shared foundation every step builds on: it loads the libraries, sets the **Inter** font with a matched output **DPI**, defines the `theme_set`/`SZ_*` design system and the env-overridable paths, and brings in the shared helper functions — so the figure is self-contained.


In [ ]:
## Locate the figure directory. Terra / Linux is case-sensitive, so match
## either "Fig2" or "fig2" and point the pipeline's env-overridable
## paths at it before sourcing the shared setup.
fig_dir <- list.files(".", pattern = "^(Fig2|fig2)$", include.dirs = TRUE)
stopifnot("Run this notebook from the repository root (the folder holding Fig2/)." =
          length(fig_dir) == 1)
Sys.setenv(FIG2_OBJ = file.path(fig_dir, "objects"),
           FIG2_DATA = file.path(fig_dir, "data"))
source(file.path(fig_dir, "00_setup.R"))
cat("Setup loaded from", fig_dir, "\n")


### Data prep

**Data prep.** Reads the raw SuSiE / QTL-burden inputs once and writes one prepared data frame per panel to `Fig2/data/<panel>.rds`. Run this before the panels (or rely on the committed `Fig2/data/`).


In [ ]:
message("Loading QTL inputs ...")
eQTL_COMB_susie <- add_derived(read_qtl("susie_eqtl"))
sQTL_COMB_susie <- add_derived(read_qtl("susie_sqtl"))
# Proteomics is optional: input_path() errors if the file is absent (no gs_uri to
# stage it), so treat that as "omit pQTL" — mirrors fig5's optional `nohom` input.
pqtl_path  <- tryCatch(input_path("susie_pqtl"), error = function(e) NA_character_)
pQTL_susie <- if (!is.na(pqtl_path)) add_derived(fread(pqtl_path, nThread = 4)) else NULL
if (is.null(pQTL_susie)) message("Note: proteomics (pQTL) input not found — omitted.")

saveit <- function(x, name) {
  saveRDS(x, file.path(DATA_DIR, paste0(name, ".rds")))
  message("  wrote ", file.path(DATA_DIR, paste0(name, ".rds")))
}

## ---- Panel A: MAF CDF of causal variants ----------------------------------
assay_tbl <- function(df, assay) df %>% transmute(variant, pip, gvs_max_af, posterior_mean, Assay = assay)
all_causal_variants <- bind_rows(
    if (!is.null(pQTL_susie)) assay_tbl(pQTL_susie, "Proteomics"),
    assay_tbl(eQTL_COMB_susie, "Expression"),
    assay_tbl(sQTL_COMB_susie, "Splicing")
  ) %>%
  filter(pip > 0.9, !is.na(gvs_max_af), gvs_max_af > 0)
xmin_box <- min(log10(all_causal_variants$gvs_max_af), na.rm = TRUE)
x_center <- mean(c(xmin_box, log10(0.01)))
saveit(list(df = all_causal_variants, x_center = x_center), "A")

## ---- Panel B: ancestry-skew, ALL fine-mapped variants (PIP > 0.9) ---------
# Ancestry-skew is variant-level (identical across assays) -> one row per variant.
skew_cols <- c("variant", "pip", "gvs_max_subpop", "gvs_max_af", "odds_ratio", "p_value", "rare")
skew_df <- bind_rows(
    eQTL_COMB_susie %>% select(all_of(skew_cols)),
    sQTL_COMB_susie %>% select(all_of(skew_cols)),
    if (!is.null(pQTL_susie)) pQTL_susie %>% select(all_of(skew_cols))
  ) %>%
  filter(pip > 0.9, !is.na(gvs_max_subpop)) %>%
  distinct(variant, .keep_all = TRUE) %>%
  mutate(padj = p.adjust(p_value, method = "fdr"),
         enriched = odds_ratio > 5 & padj < 0.05,
         pop = pretty_pop(gvs_max_subpop),
         rare_lab = if_else(rare, "Rare", "Common"))

skew_counts <- skew_df %>% filter(enriched) %>% distinct(variant, pop) %>% count(pop)
or_min      <- min(skew_df$odds_ratio[skew_df$odds_ratio > 0], na.rm = TRUE)
pair_line_df <- skew_df %>% distinct(pop) %>%
  mutate(x = as.numeric(factor(pop)) - 0.32, xend = as.numeric(factor(pop)) + 0.32,
         y = or_min * 0.55, yend = or_min * 0.55)
admix_x  <- match("ADMIX", sort(unique(skew_df$pop)))
label_df <- tibble::tibble(rare_lab = c("Common", "Rare"),
                           x = admix_x + c(-0.2, 0.2), y = or_min * 0.32)
saveit(list(skew_df = skew_df, skew_counts = skew_counts, or_min = or_min,
            pair_line_df = pair_line_df, label_df = label_df), "B")

## ---- Panel C: |log2 aFC| by variant consequence (expression) --------------
remove_list <- c(
  "stop_lost", "inframe_deletion", "stop_retained_variant", "inframe_insertion",
  "CTCF.bound", "mature_mirna_variant", "upstream_gene_variant", "intron_variant",
  "intergenic_variant", "FANTOM5", "CTCF.only", "downstream_gene_variant",
  "synonymous_variant", "start_lost", "missense_variant",
  "non_coding_transcript_variant", "non_coding_transcript_exon_variant",
  "nmd_transcript_variant", "coding_sequence_variant", "NMD_transcript_variant")
consequence_order <- c(
  "stop_gained", "frameshift_variant", "stop_lost", "start_lost",
  "splice_acceptor_variant", "splice_donor_variant", "splice_donor_5th_base_variant",
  "splice_donor_region_variant", "splice_polypyrimidine_tract_variant",
  "splice_region_variant", "PLS", "pELS", "dELS", "inframe_insertion",
  "inframe_deletion", "missense_variant", "coding_sequence_variant",
  "5_prime_UTR_variant", "3_prime_UTR_variant", "intron_variant",
  "upstream_gene_variant", "downstream_gene_variant", "intergenic_variant",
  "CTCF.bound", "CTCF.only", "DNase.H3K4me3", "FANTOM5")

worst_csq <- eQTL_COMB_susie %>%
  filter(pip > 0.9, consequence != ".") %>%
  select(variant, log2_aFC, pip, rare, posterior_mean, consequence, molecular_trait_id,
         FANTOM5, pELS, dELS, PLS, `CTCF.bound`, `DNase.H3K4me3`, `CTCF.only`) %>%
  separate_rows(consequence, sep = ",") %>%
  mutate(value = TRUE) %>%
  pivot_wider(names_from = consequence, values_from = value, values_fill = FALSE) %>%
  pivot_longer(!c(variant, log2_aFC, pip, rare, posterior_mean, molecular_trait_id)) %>%
  mutate(name = trimws(name)) %>%
  filter(!name %in% remove_list, !str_detect(name, "DNase"), value == TRUE) %>%
  mutate(consequence_rank = match(name, consequence_order)) %>%
  filter(!is.na(consequence_rank)) %>%
  group_by(variant) %>% slice_min(consequence_rank, n = 1, with_ties = FALSE) %>% ungroup() %>%
  mutate(name = if_else(str_detect(name, "splice"), "Splice region", name))

afc_plot_df <- worst_csq %>%
  mutate(abs_log2_aFC = pmin(abs(log2_aFC), 3),
         direction = if_else(log2_aFC >= 0, "Up", "Down")) %>%
  mutate(name = str_replace(name, "_prime_", "' "),
         name = str_replace(name, "UTR_variant", " UTR"),
         name = case_when(name == "stop_gained" ~ "Stop Gain",
                          name == "frameshift_variant" ~ "Frameshift", TRUE ~ name)) %>%
  mutate(name = fct_reorder(name, abs_log2_aFC, .fun = median, .desc = TRUE)) %>%
  mutate(ExpectedHet = (1 + 2^log2_aFC) / 2,
         high_effect = ExpectedHet < 0.6 | ExpectedHet > 1.4,
         color_group = paste0(name, "_", rare))

log2_afc_regression <- afc_plot_df %>%
  group_by(name) %>% do(broom::tidy(lm(abs(log2_aFC) ~ rare, data = .))) %>%
  filter(!str_detect(term, "Int")) %>% ungroup() %>%
  mutate(padj = p.adjust(p.value, method = "fdr"),
         stars = case_when(padj < 0.001 ~ "***", padj < 0.01 ~ "**", padj < 0.05 ~ "*", TRUE ~ ""),
         label = paste0("+", round(estimate, 2), stars), x = 3.1)

base_names  <- levels(afc_plot_df$name)
base_cols   <- as.character(paletteer::paletteer_d("futurevisions::pegasi")[seq_along(base_names)])
names(base_cols) <- base_names
afc_palette <- c(setNames(colorspace::darken(base_cols, 0.3),  paste0(base_names, "_TRUE")),
                 setNames(colorspace::lighten(base_cols, 0.3), paste0(base_names, "_FALSE")))
saveit(list(afc_plot_df = afc_plot_df, log2_afc_regression = log2_afc_regression,
            afc_palette = afc_palette), "C")

## ---- Panel D: cumulative expected het expression (expression) -------------
cumulative_effects <- eQTL_COMB_susie %>%
  filter(!is.na(log2_aFC)) %>%
  make_cumulative_afc_curve_both(gene_col = "molecular_trait_id")
saveit(cumulative_effects, "D")

## ---- Panel E: QTL-burden outlier enrichment -------------------------------
saveit(read_qtl("burden_enrichment"), "E")

## ---- Panel F: median genes per individual ---------------------------------
median_pc <- read_qtl("median_ko") %>%
  filter(gene_type == "protein_coding") %>%
  mutate(gene_type = "Protein Coding",
         lower = as.numeric(str_extract(as.character(pct_bin), "-?[0-9]+\\.?[0-9]*")))
saveit(median_pc, "F")

## ---- Panel G: validated dosage-like events (dumbbell) ---------------------
outliers_df <- readRDS(input_path("burden_count")) %>%
  mutate(label = str_replace(label, "\\(", "\n"),
         label = str_replace(label, ", ", "\n"),
         label = str_remove(label, "\\)"),
         label = if_else(str_detect(label, "Drug"),
                         str_replace(label, "DrugInteraction", "Drug Target"), label),
         label = fct_reorder(label, total))
saveit(outliers_df, "G")

message("Data prep complete.")


### Panel A — MAF cumulative distribution of causal variants

Cumulative minor-allele-frequency distribution of fine-mapped causal variants (fraction with MAF ≤ *k*), pooled across expression and splicing (protein where available) — the allele-frequency spectrum of causal regulatory variation.


In [ ]:
d <- readRDS(file.path(DATA_DIR, "A.rds"))

pA <- d$df %>%
  ggplot(aes(log10(gvs_max_af), color = Assay)) +
    annotate("rect", xmin = -Inf, xmax = log10(0.01), ymin = -Inf, ymax = Inf,
             alpha = 0.1, fill = "grey70") +
    stat_ecdf(linewidth = 2.5, alpha = 0.8) +
    paletteer::scale_color_paletteer_d("ltc::minou") +
    scale_x_continuous(breaks = c(-1, -2, -3), labels = c("0.1", "0.01", "0.001")) +
    scale_y_continuous(labels = percent_format(accuracy = 1)) +
    annotate("text", x = d$x_center, y = Inf, label = "Rare variants",
             color = "grey40", vjust = 1.5, size = ANN_LG, fontface = "bold") +
    labs(x = "MAF", y = "Cumulative % of causal\nvariants (MAF ≤ k)") +
    theme(legend.position = c(0.85, 0.2), axis.line.x = element_blank())

saveRDS(pA, file.path(OBJ_DIR, "A.rds"))
message("Panel A built.")

pA


### Panel B — population-enrichment distributions (all fine-mapped variants)

For every fine-mapped variant (PIP > 0.9, one row per variant) assigns a primary population and plots the distribution of population enrichment, split rare vs. common, with counts of ancestry-enriched variants — how causal variants partition across ancestries.


In [ ]:
d <- readRDS(file.path(DATA_DIR, "B.rds"))
pd <- position_dodge(width = 0.8)

pB <- d$skew_df %>%
  ggplot(aes(pop, odds_ratio, fill = pop, color = pop,
             group = interaction(pop, rare_lab))) +
    geom_hline(yintercept = 5, linetype = "dashed", linewidth = 0.5, color = "grey40") +
    geom_quasirandom(width = 0.15, dodge.width = 0.8, size = 1, alpha = 0.7, show.legend = FALSE) +
    geom_violin(width = 0.9, alpha = 0.3, position = pd, color = NA, trim = FALSE, show.legend = FALSE) +
    stat_summary(fun = median, geom = "crossbar", linewidth = 0.5, width = 0.1,
                 alpha = 0.6, position = pd, color = "black", show.legend = FALSE) +
    scale_color_carto_d(palette = "Temps") +
    scale_fill_carto_d(palette = "Temps") +
    geom_text(data = d$skew_counts, aes(pop, 9000, label = scales::comma(n)),
              inherit.aes = FALSE, size = ANN_LG, color = "grey40", fontface = "bold") +
    geom_segment(data = d$pair_line_df, aes(x = x, xend = xend, y = y, yend = yend),
                 inherit.aes = FALSE, color = "grey60", linewidth = 1) +
    geom_text(data = d$label_df, aes(x = x, y = y, label = rare_lab),
              inherit.aes = FALSE, size = ANN_SM, color = "black", angle = 30) +
    annotate("segment", x = 0.85, xend = 7.15, y = 18000, yend = 18000,
             linewidth = 0.5, color = "grey40") +
    annotate("text", x = 4, y = 26000, label = "Ancestry-enriched variants",
             size = ANN_LG, fontface = "bold", color = "grey40") +
    scale_y_continuous(trans = "log10", breaks = c(1, 10, 100, 1000, 10000),
                       labels = scales::trans_format("log10", scales::math_format(10^.x))) +
    labs(x = "Primary Population", y = "Causal variant population enrichment") +
    theme(axis.line.x = element_blank(), axis.ticks.x = element_blank())

saveRDS(pB, file.path(OBJ_DIR, "B.rds"))
message("Panel B built.")

pB


### Panel C — |log2 aFC| effect size by variant consequence (expression)

Effect size (|log2 allelic fold-change|) by variant molecular consequence, rare (dark) vs. common (light), annotated with the rare−common effect shift per consequence — do rare causal variants carry larger effects?


In [ ]:
d <- readRDS(file.path(DATA_DIR, "C.rds"))

pC <- d$afc_plot_df %>%
  filter(!str_detect(color_group, "NA")) %>%
  ggplot(aes(abs_log2_aFC, name, color = color_group, fill = color_group)) +
    geom_vline(xintercept = 1, linetype = "dashed") +
    stat_halfeye(fill_type = "segments", point_interval = "median_qi",
                 show.legend = FALSE, height = 1, point_size = 6,
                 interval_size = 0.4, slab_linewidth = 5.5, slab_alpha = 0.7) +
    scale_fill_manual(values = d$afc_palette) +
    scale_color_manual(values = d$afc_palette) +
    geom_text(data = d$log2_afc_regression, aes(x, name, label = label),
              inherit.aes = FALSE, hjust = 0, size = ANN_SM) +
    labs(x = "|Log2 aFC|", y = NULL) +
    coord_cartesian(clip = "off") +
    theme(axis.line.y = element_blank(), axis.ticks.y = element_blank(),
          plot.margin = margin(5.5, 45, 5.5, 5.5))

saveRDS(pC, file.path(OBJ_DIR, "C.rds"))
message("Panel C built.")

pC


### Panel D — expected heterozygous expression from cumulative aFC (expression)

Cumulatively combines a gene's independent causal variants to estimate expected heterozygous expression, contrasting deletion-like and duplication-like directions — a dosage model built from the fine-mapped effects.


In [ ]:
cumulative_effects <- readRDS(file.path(DATA_DIR, "D.rds"))

pD <- PlotExpectedExpression(cumulative_effects)

saveRDS(pD, file.path(OBJ_DIR, "D.rds"))
message("Panel D built.")

pD


### Panel E — QTL-burden outlier enrichment

Expression-outlier enrichment across effect-size bins (under- vs. over-expression), from the QTL-burden summary — linking causal-variant burden to outlier expression.


In [ ]:
OutlierEnrichments <- readRDS(file.path(DATA_DIR, "E.rds"))
outlier_pal <- c("#D44D5CFF", "#046E8FFF")

pE <- OutlierEnrichments %>%
  filter(focal_lower_bound != -10) %>%
  ggplot(aes(focal_lower_bound, odds_ratio, color = type, fill = type, group = type)) +
    geom_hline(yintercept = 1, linetype = "dashed") +
    geom_line(show.legend = FALSE) +
    geom_linerange(aes(ymin = ci_low, ymax = ci_high), show.legend = FALSE) +
    geom_point(size = 6, shape = 21, alpha = 0.7, show.legend = FALSE) +
    scale_fill_manual(values = outlier_pal) +
    scale_color_manual(values = outlier_pal) +
    scale_y_continuous(trans = "log2", breaks = 2^c(-4, 0, 4, 8),
                       labels = function(x) parse(text = paste0("2^", log2(x)))) +
    annotate("text", x = -100, y = 7, label = "Underlier", color = "#D44D5CFF", size = ANN_SM) +
    annotate("text", x =  175, y = 4, label = "Overlier",  color = "#046E8FFF", size = ANN_SM) +
    labs(y = "Outlier Enrichment") +
    theme(axis.line.x = element_blank(), axis.ticks.x = element_blank(),
          axis.text.x = element_blank(), axis.title.x = element_blank())

saveRDS(pE, file.path(OBJ_DIR, "E.rds"))
message("Panel E built.")

pE


### Panel F — median genes per individual by percent-change bin

Median number of dosage-affected protein-coding genes per individual by percent-change bin, with the deletion-/duplication-like bins highlighted.


In [ ]:
median_pc <- readRDS(file.path(DATA_DIR, "F.rds"))

# Grey highlight boxes over the deletion-like / duplication-like bins.
DelBoxStart <- 0.5;  DelBoxEnd <- 2.45
DupBoxStart <- 7.5;  DupBoxEnd <- 13.45
DupBoxMid   <- DupBoxStart + (DupBoxEnd - DupBoxStart) / 2
max_median  <- max(median_pc$q75_genes)

pF <- median_pc %>%
  ggplot(aes(reorder(pct_bin, lower), median_genes)) +
    geom_rect(xmin = DelBoxStart, xmax = DelBoxEnd, ymin = -1, ymax = max_median + 100,
              inherit.aes = FALSE, fill = "lightgrey", alpha = 0.05) +
    geom_rect(xmin = DupBoxStart, xmax = DupBoxEnd, ymin = -1, ymax = max_median + 100,
              inherit.aes = FALSE, fill = "lightgrey", alpha = 0.05) +
    geom_col(width = 0.8, alpha = 0.6, linewidth = 0.4, color = "#462255FF", fill = "#462255FF") +
    geom_errorbar(aes(ymin = q25_genes, ymax = q75_genes), alpha = 0.7, width = 0.2) +
    facet_wrap(~gene_type, nrow = 2, strip.position = "right") +
    geom_text(x = 1.5,       y = max_median + 250, label = "Del-Like", size = ANN_MD,
              color = "grey35", fontface = "bold") +
    geom_text(x = DupBoxMid, y = max_median + 250, label = "Dup-Like", size = ANN_MD,
              color = "grey35", fontface = "bold") +
    geom_segment(aes(x = 0.6, xend = 2.25, y = 1000, yend = 1000),
                 inherit.aes = FALSE, color = "black", linewidth = 0.25) +
    geom_text(x = 1.425, y = 1200, label = "5 Genes", size = ANN_MD,
              color = "grey35", fontface = "bold") +
    geom_segment(aes(x = DupBoxStart + 0.5, xend = DupBoxEnd - 0.5, y = 1000, yend = 1000),
                 inherit.aes = FALSE, color = "black", linewidth = 0.25) +
    geom_text(x = DupBoxMid, y = 1200, label = "108 Genes", size = ANN_MD,
              color = "grey35", fontface = "bold") +
    labs(x = "Percent change bin", y = "Median genes per individual") +
    theme(axis.line.x = element_blank(), axis.ticks.x = element_blank(),
          axis.text.x = element_text(angle = 90))

saveRDS(pF, file.path(OBJ_DIR, "F.rds"))
message("Panel F built.")

pF


### Panel G — validated dosage-like events (dumbbell)

Counts of validated dosage-like events, deletion- vs. duplication-like, drawn as a dumbbell (segment + two end points, no extra dependency).


In [ ]:
outliers_df <- readRDS(file.path(DATA_DIR, "G.rds"))
outlier_pal2 <- c("Expression Underlier" = "#D44D5CFF", "Expression Overlier" = "#046E8FFF")

# Dumbbell drawn manually (segment + two end points) — no ggalt dependency.
pG <- ggplot(outliers_df, aes(y = label)) +
  geom_segment(aes(x = `Expression Underlier`, xend = `Expression Overlier`, yend = label),
               color = "grey65", linewidth = 2) +
  geom_point(aes(x = `Expression Underlier`), color = outlier_pal2["Expression Underlier"], size = 10) +
  geom_point(aes(x = `Expression Overlier`),  color = outlier_pal2["Expression Overlier"],  size = 10) +
  geom_text(aes(x = `Expression Underlier`, label = scales::comma(`Expression Underlier`)),
            hjust = 1.35, size = ANN_MD) +
  geom_text(aes(x = `Expression Overlier`, label = scales::comma(`Expression Overlier`)),
            hjust = -0.5, size = ANN_MD) +
  scale_x_continuous(labels = scales::comma, expand = expansion(mult = c(0.08, 0.18))) +
  annotate("point", x = 1850, y = 0.75, color = "#D44D5CFF", size = 5) +
  annotate("text",  x = 2000, y = 0.75, label = "Deletion-like",   hjust = 0, size = ANN_MD) +
  annotate("point", x = 1850, y = 0.55, color = "#046E8FFF", size = 5) +
  annotate("text",  x = 2000, y = 0.55, label = "Duplication-like", hjust = 0, size = ANN_MD) +
  labs(x = "Validated dosage-like events", y = NULL) +
  theme(axis.line.y = element_blank(), axis.ticks.y = element_blank(),
        axis.text.y = element_text(color = "black", margin = margin(r = 25)),
        plot.margin = margin(t = 6, r = 10, b = 6, l = 4))

saveRDS(pG, file.path(OBJ_DIR, "G.rds"))
message("Panel G built.")

pG


## Assemble the figure

Loads the cached panel objects from `Fig2/objects/` and composes them with patchwork into the final figure. Panel **tagging happens here and only here**. Writes the PDF plus a PNG preview, and renders the assembled figure inline.


In [ ]:
P <- setNames(lapply(LETTERS[1:7], function(x) readRDS(file.path(OBJ_DIR, paste0(x, ".rds")))),
              LETTERS[1:7])

# E over F form the QTL-burden column.
QTLBurden <- P$E / P$F + plot_layout(heights = c(1, 3))

top_row <- P$A + (P$B + theme(plot.margin = margin(l = 100))) + P$C +
  plot_layout(widths = c(1, 2.5, 1))

Fig2 <- top_row / (P$D | QTLBurden | P$G) +
  plot_annotation(tag_levels = "A") &
  theme(plot.tag = element_text(size = SZ_TAG))

ggsave(file.path(OUTDIR, "Fig2.pdf"), Fig2, width = 40, height = 24, dpi = DPI, limitsize = FALSE)
# Raster preview: full physical size at lower DPI (showtext DPI matched).
showtext_opts(dpi = 100)
ggsave(file.path(OUTDIR, "Fig2.png"), Fig2, width = 40, height = 24, dpi = 100, limitsize = FALSE)
showtext_opts(dpi = DPI)

message("Assembled: ", file.path(OUTDIR, "Fig2.pdf"), " and Fig2.png")

Fig2


## **Supplemental (ancestry composition).** Composition of ancestry-enriched fine-mapped expression variants across AoU allele frequency, stacked by enriched ancestry, with the proportion shared across all continents overlaid.


In [ ]:
message("Loading expression input ...")
eQTL_COMB_susie <- add_derived(read_qtl("susie_eqtl"))

af_cols <- c("gvs_all_af", "gvs_max_af", "gvs_afr_af", "gvs_amr_af",
             "gvs_eas_af", "gvs_eur_af", "gvs_mid_af", "gvs_sas_af", "gvs_oth_af")

AncestrySkewAnnotated <- eQTL_COMB_susie %>%
  filter(pip > 0.9) %>%                        # fine-mapped variants only
  mutate(across(all_of(af_cols), as.numeric)) %>%
  mutate(
    MAF     = if_else(gvs_all_af > 0.5, 1 - gvs_all_af, gvs_all_af),
    log_MAF = -log10(MAF),
    padj    = p.adjust(p_value, method = "fdr"),
    af_bin  = ntile(log_MAF, 30),
    enriched = odds_ratio > 5 & padj < 0.05,
    present_afr = gvs_afr_af > 0, present_eur = gvs_eur_af > 0,
    present_eas = gvs_eas_af > 0, present_amr = gvs_amr_af > 0,
    present_sas = gvs_sas_af > 0, present_mid = gvs_mid_af > 0,
    present_oth = gvs_oth_af > 0
  ) %>%
  mutate(shared_all_continents = present_afr & present_eur & present_eas &
           present_amr & present_mid & present_sas & present_oth)

# AF bins -> representative x (geometric mean of the bin's AF range).
bin_key <- AncestrySkewAnnotated %>%
  group_by(af_bin) %>%
  summarise(min_log = min(log_MAF, na.rm = TRUE),
            max_log = max(log_MAF, na.rm = TRUE), .groups = "drop") %>%
  mutate(min_af = 10^(-max_log), max_af = 10^(-min_log), x = sqrt(min_af * max_af)) %>%
  select(af_bin, x)

all_pops   <- sort(unique(AncestrySkewAnnotated$gvs_max_subpop))
bin_totals <- AncestrySkewAnnotated %>% count(af_bin, name = "total_variants")

# Enriched-variant composition by ancestry, per AF bin.
stack_df <- AncestrySkewAnnotated %>%
  filter(enriched) %>%
  count(af_bin, gvs_max_subpop, name = "n_enriched") %>%
  complete(af_bin = bin_key$af_bin, gvs_max_subpop = all_pops, fill = list(n_enriched = 0)) %>%
  left_join(bin_totals, by = "af_bin") %>%
  mutate(total_variants = replace_na(total_variants, 0),
         prop = if_else(total_variants > 0, n_enriched / total_variants, 0)) %>%
  left_join(bin_key, by = "af_bin")

# Proportion of variants shared across all continents, per AF bin (smoothed).
line_df <- AncestrySkewAnnotated %>%
  group_by(af_bin) %>%
  summarise(prop_shared = mean(shared_all_continents, na.rm = TRUE), .groups = "drop") %>%
  complete(af_bin = bin_key$af_bin, fill = list(prop_shared = 0)) %>%
  left_join(bin_key, by = "af_bin") %>%
  arrange(x) %>%
  mutate(prop_shared_smooth = slide_dbl(prop_shared, mean, .before = 2, .after = 2, .complete = FALSE))

# Save the data for an assembly/re-plot script.
saveRDS(list(stack_df = stack_df, line_df = line_df),
        file.path(DATA_DIR, "S_ancestry_stacked.rds"))

stacked_histogram <- stack_df %>%
  mutate(pop = pretty_pop(gvs_max_subpop)) %>%
  ggplot(aes(x, prop, fill = pop)) +
    geom_area(position = "stack", alpha = 0.95) +
    geom_smooth(inherit.aes = FALSE, data = line_df, aes(x, prop_shared_smooth),
                color = "black", linewidth = 0.4, se = FALSE) +
    geom_vline(xintercept = 0.5, color = "white") +
    scale_fill_carto_d(palette = "Temps") +
    scale_x_log10(breaks = c(1e-4, 1e-3, 1e-2, 1e-1, 5e-1),
                  labels = c("0.0001", "0.001", "0.01", "0.1", "0.5"), expand = c(0, 0)) +
    scale_y_continuous(labels = percent_format(accuracy = 1),
                       expand = c(0, 0.001), breaks = c(.25, .5, .75, 1)) +
    labs(x = "AoU allele frequency",
         y = "Proportion of ancestry-enriched variants",
         fill = "Enriched\nancestry") +
    theme(axis.line = element_line(color = "black", linewidth = 0.3),
          axis.ticks = element_line(linewidth = 0.2),
          axis.title.y = element_text(margin = margin(r = 20)),
          legend.position = c(0.9, 0.5))

saveRDS(stacked_histogram, file.path(OBJ_DIR, "S_ancestry_stacked.rds"))

# Standalone supplemental figure (single panel; sized like a main-figure panel
# so the shared text sizes read correctly).
ggsave(file.path(OUTDIR, "FigS_AncestryStacked.pdf"), stacked_histogram,
       width = 13, height = 9, dpi = DPI)
showtext_opts(dpi = 100)
ggsave(file.path(OUTDIR, "FigS_AncestryStacked.png"), stacked_histogram,
       width = 13, height = 9, dpi = 100)
showtext_opts(dpi = DPI)

message("Wrote FigS_AncestryStacked.pdf / .png")

stacked_histogram
